# Gestión de DataFrames con Pandas: operaciones y transformaciones

**Módulo:** Programación de Inteligencia Artificial (PIA) — Curso de Especialización en IA y Big Data
**I.E.S. Ribera de Castilla** · Curso 2026-27

---

Estos apuntes recorren las operaciones más habituales que se aplican sobre un `DataFrame` una vez cargados los datos: cómo organizar el índice, limpiar valores ausentes y duplicados, transformar texto, combinar tablas, reestructurar su forma y resumir la información por grupos.

> **Referencia:** Bitwright, C. *Data Manipulation with Pandas: Sculpting Raw Information into Actionable Insight* (2026), capítulos 5 a 11.    
Los ejemplos se han adaptado y actualizado a **pandas 3.x**: allí donde versiones anteriores usaban sintaxis obsoleta se indica con una nota ⚠️.

### Índice

1. [Indexación avanzada y MultiIndex](#1.-Indexación-avanzada-y-MultiIndex) *(cap. 5)*
2. [Tratamiento de datos ausentes](#2.-Tratamiento-de-datos-ausentes) *(cap. 6)*
3. [Depuración y transformación de datos](#3.-Depuración-y-transformación-de-datos) *(cap. 7)*
4. [Operaciones vectorizadas con cadenas](#4.-Operaciones-vectorizadas-con-cadenas) *(cap. 8)*
5. [Combinación de DataFrames: concat, merge y join](#5.-Combinación-de-DataFrames:-concat,-merge-y-join) *(cap. 9)*
6. [Reestructuración: pivot, melt, stack y unstack](#6.-Reestructuración:-pivot,-melt,-stack-y-unstack) *(cap. 10)*
7. [Agrupación y agregación: Split-Apply-Combine](#7.-Agrupación-y-agregación:-Split-Apply-Combine) *(cap. 11)*
8. [Ejercicios propuestos](#8.-Ejercicios-propuestos)

In [1]:
import numpy as np
import pandas as pd

print("pandas", pd.__version__)

pandas 3.0.6


> **Recordatorio — los métodos devuelven copias.** Casi todos los métodos de pandas (`set_index`, `dropna`, `rename`, `fillna`…) **no modifican** el DataFrame original, sino que devuelven uno nuevo. Para conservar el cambio hay que reasignarlo:
> ```python
> df = df.dropna()          # recomendado
> df.dropna(inplace=True)   # existe, pero se desaconseja: no aporta rendimiento y rompe el encadenamiento de métodos
> ```

## 1. Indexación avanzada y MultiIndex

El índice numérico por defecto (`0, 1, 2…`) es cómodo, pero muchas veces el identificador natural de cada fila ya está en una columna (un código de cliente, una fecha…). Además, cuando los datos tienen agrupaciones naturales (trimestre → región → cliente), un **índice jerárquico** permite representarlos de forma ordenada en dos dimensiones.

### 1.1 Establecer y restablecer el índice

| Método | Qué hace |
|---|---|
| `df.set_index(col)` | Convierte una o varias columnas en el índice de filas |
| `df.reset_index()` | Operación inversa: devuelve el índice (o niveles del mismo) a columnas |

In [2]:
ventas = pd.DataFrame({
    "id": [101, 102, 103],
    "trimestre": ["T1", "T1", "T2"],
    "ingresos": [5000, 6200, 7100],
})

ventas_idx = ventas.set_index("id")   # 'id' pasa a ser el índice
ventas_idx

,trimestre,ingresos
id,,
101,T1,5000
102,T1,6200
103,T2,7100


In [3]:
ventas_idx.reset_index()   # 'id' vuelve a ser una columna

,id,trimestre,ingresos
0,101,T1,5000
1,102,T1,6200
2,103,T2,7100


> Si el índice no tenía nombre, `reset_index()` lo convierte en una columna llamada `index`. Con `reset_index(drop=True)` el índice se descarta en lugar de pasar a columna (útil tras filtrar filas para renumerar).

### 1.2 Crear un índice jerárquico (MultiIndex)

Basta con pasar una **lista de columnas** a `set_index()`. El primer elemento será el nivel exterior y el último el interior.

In [4]:
ventas_multi = ventas.set_index(["trimestre", "id"])
ventas_multi

ingresos
trimestre id           
T1        101      5000
          102      6200
T2        103      7100

In [5]:
ventas_multi.index            # objeto MultiIndex

MultiIndex([('T1', 101),
            ('T1', 102),
            ('T2', 103)],
           names=['trimestre', 'id'])

In [6]:
ventas_multi.index.names, ventas_multi.index.nlevels

(FrozenList(['trimestre', 'id']), 2)

### 1.3 Selección de datos en un MultiIndex

Se utiliza `.loc[]`, indicando las etiquetas de cada nivel mediante **tuplas**.

In [7]:
# Por el nivel exterior: devuelve todas las filas de 'T1'
# (el nivel 'trimestre' desaparece del resultado)
ventas_multi.loc["T1"]

,ingresos
id,
101,5000
102,6200


In [8]:
# Combinación concreta de niveles: tupla (trimestre, id)
ventas_multi.loc[("T2", 103)]

ingresos    7100
Name: (T2, 103), dtype: int64

**Selección por niveles interiores con `pd.IndexSlice`**

Para hacer *slicing* sobre niveles interiores, la forma explícita es `pd.IndexSlice`. El índice debe estar **ordenado** (`sort_index()`).

In [9]:
idx = pd.IndexSlice
ventas_multi.sort_index().loc[idx[:, 102:103], :]   # cualquier trimestre, id entre 102 y 103

,,ingresos
trimestre,id,
T1,102,6200
T2,103,7100


### 1.4 Intercambiar y reordenar niveles

| Método | Uso |
|---|---|
| `df.swaplevel(a, b)` | Intercambia dos niveles concretos |
| `df.reorder_levels([...])` | Establece un orden arbitrario para todos los niveles |
| `df.sort_index()` | Reordena las filas según el nuevo orden de niveles |

In [10]:
df = pd.DataFrame({
    "id": [101, 102, 103, 104],
    "trimestre": ["T1", "T1", "T2", "T2"],
    "region": ["Este", "Oeste", "Este", "Oeste"],
    "ingresos": [5000, 6200, 7100, 5800],
}).set_index(["trimestre", "region", "id"])
df

ingresos
trimestre region id           
T1        Este   101      5000
          Oeste  102      6200
T2        Este   103      7100
          Oeste  104      5800

In [11]:
# Ahora la región pasa a ser el nivel exterior
df_swap = df.swaplevel("trimestre", "region")
df_swap.sort_index()

ingresos
region trimestre id           
Este   T1        101      5000
       T2        103      7100
Oeste  T1        102      6200
       T2        104      5800

In [12]:
df_swap.reorder_levels(["id", "region", "trimestre"])

,,,ingresos
id,region,trimestre,
101,Este,T1,5000
102,Oeste,T1,6200
103,Este,T2,7100
104,Oeste,T2,5800


> Ni `swaplevel` ni `reorder_levels` alteran los datos: sólo cambian la jerarquía de etiquetas. Suele ser conveniente encadenar `.sort_index()` después para que las filas queden agrupadas según el nuevo nivel exterior.

### 1.5 Secciones transversales con `xs()`

`df.xs(clave, level=nivel, axis=0)` selecciona todas las filas que tienen un valor dado **en cualquier nivel**, sin tener que especificar los niveles exteriores. Es la forma más legible de filtrar por un nivel interior.

In [13]:
df.xs("T1", level="trimestre")     # equivalente a df.loc["T1"]

,,ingresos
region,id,
Este,101,5000
Oeste,102,6200


In [14]:
# Aquí está la ventaja: filtrar por un nivel interior
df.xs("Este", level="region")

,,ingresos
trimestre,id,
T1,101,5000
T2,103,7100


`xs()` también funciona sobre columnas jerárquicas con `axis=1` (lo veremos en el apartado 6, donde las tablas dinámicas generan columnas con MultiIndex).

## 2. Tratamiento de datos ausentes

Los datos reales casi nunca están completos. En pandas los valores ausentes se representan como `NaN` (*Not a Number*), `None` o `pd.NA`, y deben tratarse antes de cualquier análisis o modelo.

Estrategias:
1. **Detectar** dónde están (`isna`, `notna`).
2. **Eliminar** filas/columnas (`dropna`) si son pocos.
3. **Imputar** (rellenar) con un valor razonable (`fillna`, `ffill`, `bfill`, `interpolate`).

### 2.1 Detección: `isna()` y `notna()`

Devuelven una máscara booleana del mismo tamaño que el DataFrame.

In [15]:
clientes = pd.DataFrame({
    "cliente_id": [1, 2, 3, 4, 5],
    "edad": [30, 45, np.nan, 22, 50],
    "ingresos": [60000, 75000, 80000, np.nan, 95000],
    "producto_pref": ["A", "B", "A", "C", np.nan],
})
clientes

,cliente_id,edad,ingresos,producto_pref
0,1,30.0,60000.0,A
1,2,45.0,75000.0,B
2,3,NaN,80000.0,A
3,4,22.0,NaN,C
4,5,50.0,95000.0,NaN


In [16]:
clientes.isna()

,cliente_id,edad,ingresos,producto_pref
0,False,False,False,False
1,False,False,False,False
2,False,True,False,False
3,False,False,True,False
4,False,False,False,True


In [17]:
# Recuento de ausentes por columna (True cuenta como 1)
clientes.isna().sum()

cliente_id       0
edad             1
ingresos         1
producto_pref    1
dtype: int64

In [18]:
# Porcentaje de ausentes por columna, muy útil en un EDA
clientes.isna().mean() * 100

cliente_id        0.0
edad             20.0
ingresos         20.0
producto_pref    20.0
dtype: float64

### 2.2 Eliminación: `dropna()`

Sintaxis: `df.dropna(axis=0, how='any', thresh=None, subset=None)`

| Parámetro | Significado |
|---|---|
| `axis=0` / `axis=1` | Eliminar filas (por defecto) o columnas |
| `how='any'` | Elimina si hay **algún** NaN (estricto, por defecto) |
| `how='all'` | Elimina sólo si **todos** son NaN (permisivo) |
| `thresh=n` | Conserva las filas con al menos `n` valores **no nulos** |
| `subset=[...]` | Sólo considera estas columnas |

In [19]:
clientes.dropna()                 # filas con algún NaN fuera

,cliente_id,edad,ingresos,producto_pref
0,1,30.0,60000.0,A
1,2,45.0,75000.0,B


In [20]:
clientes.dropna(axis=1)           # columnas con algún NaN fuera

,cliente_id
0,1
1,2
2,3
3,4
4,5


In [21]:
clientes.dropna(thresh=4)         # al menos 4 valores válidos por fila

,cliente_id,edad,ingresos,producto_pref
0,1,30.0,60000.0,A
1,2,45.0,75000.0,B


In [22]:
clientes.dropna(subset=["edad"])  # sólo importan los NaN de 'edad'

,cliente_id,edad,ingresos,producto_pref
0,1,30.0,60000.0,A
1,2,45.0,75000.0,B
3,4,22.0,NaN,C
4,5,50.0,95000.0,NaN


> Eliminar es sencillo pero puede **perder información y sesgar** los datos restantes. Por eso suele preferirse imputar.

### 2.3 Imputación con `fillna()`

La técnica depende de la naturaleza de la variable:

| Tipo de dato | Imputación habitual |
|---|---|
| Categórica | Constante (`'Desconocido'`) o la moda |
| Numérica ~normal | Media |
| Numérica asimétrica / con atípicos | Mediana (robusta frente a valores extremos) |

In [23]:
imp = clientes.copy()

imp["producto_pref"] = imp["producto_pref"].fillna("Desconocido")   # constante
imp["edad"] = imp["edad"].fillna(imp["edad"].mean())                # media: (30+45+22+50)/4 = 36.75
imp["ingresos"] = imp["ingresos"].fillna(imp["ingresos"].median())  # mediana: (75000+80000)/2 = 77500
imp

,cliente_id,edad,ingresos,producto_pref
0,1,30.00,60000.0,A
1,2,45.00,75000.0,B
2,3,36.75,80000.0,A
3,4,22.00,77500.0,C
4,5,50.00,95000.0,Desconocido


También se puede pasar a `fillna()` un **diccionario** con un valor por columna:

In [24]:
clientes.fillna({"producto_pref": "Desconocido",
                 "edad": clientes["edad"].median()})

,cliente_id,edad,ingresos,producto_pref
0,1,30.0,60000.0,A
1,2,45.0,75000.0,B
2,3,37.5,80000.0,A
3,4,22.0,NaN,C
4,5,50.0,95000.0,Desconocido


**Imputación por grupos.** A menudo una media global no es adecuada (p. ej., la edad media difiere entre regiones). La solución es combinar `groupby()` + `transform()` + `fillna()`; se verá en el apartado 7.4.

### 2.4 Relleno hacia delante y hacia atrás: `ffill()` y `bfill()`

En datos **secuenciales** (series temporales, lecturas de sensores, cotizaciones), el mejor estimador de un valor ausente suele ser el valor contiguo:

- **`ffill()`** (*forward fill*): propaga el último valor válido hacia delante.
- **`bfill()`** (*backward fill*): propaga el siguiente valor válido hacia atrás.

> ⚠️ **Cambio respecto a versiones anteriores:** en versiones anteriores se usaba la forma `df.fillna(method='ffill')`. Esa forma está **eliminada en pandas 3**; se usan directamente los métodos `df.ffill()` y `df.bfill()`.

In [25]:
sensor = pd.DataFrame({
    "fecha": pd.to_datetime(["2025-01-01", "2025-01-02", "2025-01-03",
                             "2025-01-04", "2025-01-05"]),
    "lectura": [10.5, np.nan, 12.0, np.nan, np.nan],
}).set_index("fecha")
sensor

,lectura
fecha,
2025-01-01,10.5
2025-01-02,NaN
2025-01-03,12.0
2025-01-04,NaN
2025-01-05,NaN


In [26]:
sensor.ffill()     # 02 → 10.5 ; 04 y 05 → 12.0

,lectura
fecha,
2025-01-01,10.5
2025-01-02,10.5
2025-01-03,12.0
2025-01-04,12.0
2025-01-05,12.0


In [27]:
sensor.bfill()     # 02 → 12.0 ; 04 y 05 siguen NaN (no hay valor posterior)

,lectura
fecha,
2025-01-01,10.5
2025-01-02,12.0
2025-01-03,12.0
2025-01-04,NaN
2025-01-05,NaN


In [28]:
sensor.ffill(limit=1)   # 'limit' restringe cuántos huecos consecutivos se rellenan

,lectura
fecha,
2025-01-01,10.5
2025-01-02,10.5
2025-01-03,12.0
2025-01-04,12.0
2025-01-05,NaN


### 2.5 Interpolación: `interpolate()`

`ffill`/`bfill` suponen que el valor cambia "a saltos". Si el fenómeno es **continuo**, la interpolación estima el valor ajustando una curva entre los puntos conocidos.

| `method=` | Uso |
|---|---|
| `'linear'` (por defecto) | Recta entre los puntos vecinos, ignorando el índice |
| `'time'` | Tiene en cuenta la distancia temporal real del `DatetimeIndex` |
| `'polynomial'`, `'spline'` | Curvas más suaves (requieren `order=` y SciPy) |

In [29]:
sensor.interpolate()    # 02 → (10.5 + 12.0)/2 = 11.25 ; al final no hay punto posterior → repite 12.0

,lectura
fecha,
2025-01-01,10.50
2025-01-02,11.25
2025-01-03,12.00
2025-01-04,12.00
2025-01-05,12.00


In [30]:
# Con fechas irregulares, 'time' pondera por la distancia real
irregular = pd.Series([1.0, np.nan, 4.0],
                      index=pd.to_datetime(["2025-01-01", "2025-01-02", "2025-01-05"]))
pd.DataFrame({"linear": irregular.interpolate(),
              "time": irregular.interpolate(method="time")})

,linear,time
2025-01-01,1.0,1.00
2025-01-02,2.5,1.75
2025-01-05,4.0,4.00


## 3. Depuración y transformación de datos

El *data wrangling* consiste en dejar los datos limpios y homogéneos: nombres de columnas coherentes, tipos de datos correctos, sin duplicados y con categorías estandarizadas.

### 3.1 Renombrar etiquetas: `rename()`

Se pasa un diccionario `{nombre_antiguo: nombre_nuevo}` a `columns=` o a `index=`.

In [31]:
pedidos = pd.DataFrame({
    "Customer Id": [1, 2, 3],
    "product_id": ["A1", "B2", "C3"],
    "Revenue (USD)": [100.5, 200.2, 50.8],
}, index=["fila_0", "fila_1", "fila_2"])
pedidos

,Customer Id,product_id,Revenue (USD)
fila_0,1,A1,100.5
fila_1,2,B2,200.2
fila_2,3,C3,50.8


In [32]:
pedidos_ren = pedidos.rename(columns={"Customer Id": "cliente_id",
                                      "product_id": "producto_id",
                                      "Revenue (USD)": "ingresos_usd"})
pedidos_ren = pedidos_ren.rename(index={"fila_0": "primera"})
pedidos_ren

,cliente_id,producto_id,ingresos_usd
primera,1,A1,100.5
fila_1,2,B2,200.2
fila_2,3,C3,50.8


`rename()` también admite una **función**, que se aplica a todas las etiquetas. Es la forma rápida de normalizar todos los nombres a la vez:

In [33]:
pedidos.rename(columns=lambda c: c.lower().replace(" ", "_"))

,customer_id,product_id,revenue_(usd)
fila_0,1,A1,100.5
fila_1,2,B2,200.2
fila_2,3,C3,50.8


### 3.2 Tipos de datos: `dtypes` y `astype()`

- `df.dtypes` (**atributo**, sin paréntesis) muestra el tipo de cada columna.
- `serie.astype(tipo)` convierte explícitamente a otro tipo.

| Tipo | Uso |
|---|---|
| `int64`, `float64` | Numéricos estándar |
| `int32`, `float32` | Numéricos con menos memoria |
| `bool` | Verdadero/falso |
| `str` | Texto (en pandas 3 es el tipo por defecto para cadenas) |
| `object` | Objetos Python genéricos o tipos mezclados |
| `category` | Pocos valores distintos repetidos: muy eficiente en memoria |

> ⚠️ **Novedad de pandas 3:** las columnas de texto ya no se cargan como `object`, sino con el tipo de cadena dedicado `str`.

In [34]:
pedidos.dtypes

Customer Id        int64
product_id           str
Revenue (USD)    float64
dtype: object

In [35]:
conv = pedidos.copy()
conv["Customer Id"] = conv["Customer Id"].astype("str")          # entero → texto
conv["Revenue (USD)"] = conv["Revenue (USD)"].astype("float32")  # menos memoria
conv["product_id"] = conv["product_id"].astype("category")
conv.dtypes

Customer Id           str
product_id       category
Revenue (USD)     float32
dtype: object

Cuando una columna de texto contiene números "sucios", `astype()` falla. En ese caso se usa `pd.to_numeric(..., errors='coerce')`, que convierte en `NaN` lo que no se puede interpretar:

In [36]:
pd.to_numeric(pd.Series(["10", "3.5", "n/d", "7"]), errors="coerce")

0    10.0
1     3.5
2     NaN
3     7.0
dtype: float64

### 3.3 Eliminar duplicados: `duplicated()` y `drop_duplicates()`

Los registros duplicados (habituales al juntar fuentes) inflan las métricas y sesgan los modelos.

- `duplicated()` → máscara booleana: `True` en las filas que repiten una anterior.
- `drop_duplicates()` → elimina esas filas.

| Parámetro | Significado |
|---|---|
| `keep='first'` | Conserva la primera aparición (por defecto) |
| `keep='last'` | Conserva la última |
| `keep=False` | Marca/elimina **todas** las apariciones |
| `subset=[...]` | Define la duplicidad sólo con estas columnas |

In [37]:
ordenes = pd.DataFrame({
    "orden_id": [101, 102, 101, 103, 102],
    "producto_id": ["P5", "P8", "P5", "P9", "P8"],
    "importe": [50.0, 75.0, 50.0, 90.0, 75.0],
})
ordenes.duplicated()

0    False
1    False
2     True
3    False
4     True
dtype: bool

In [38]:
ordenes.duplicated(keep=False)    # marca todas las copias, también la primera

0     True
1     True
2     True
3    False
4     True
dtype: bool

In [39]:
ordenes.drop_duplicates()

,orden_id,producto_id,importe
0,101,P5,50.0
1,102,P8,75.0
3,103,P9,90.0


In [40]:
ordenes.drop_duplicates(subset=["orden_id"], keep="last")

,orden_id,producto_id,importe
2,101,P5,50.0
3,103,P9,90.0
4,102,P8,75.0


### 3.4 Mapear y reemplazar valores: `map()` y `replace()`

| Método | Se aplica a | Comportamiento con valores no previstos |
|---|---|---|
| `serie.map(dict / función)` | Series | Lo que no está en el diccionario → `NaN` |
| `df.replace(...)` | Series o DataFrame | Lo que no se menciona **se queda igual** |

In [41]:
nombres = {"P5": "Escáner", "P8": "Proyector"}       # falta 'P9' a propósito
ordenes["producto"] = ordenes["producto_id"].map(nombres)
ordenes

,orden_id,producto_id,importe,producto
0,101,P5,50.0,Escáner
1,102,P8,75.0,Proyector
2,101,P5,50.0,Escáner
3,103,P9,90.0,NaN
4,102,P8,75.0,Proyector


In [42]:
# map() también acepta funciones
ordenes["importe"].map(lambda x: "alto" if x >= 75 else "bajo")

0    bajo
1    alto
2    bajo
3    alto
4    alto
Name: importe, dtype: str

In [43]:
# replace(): corregir un valor en una columna concreta…
ordenes.replace({"producto_id": {"P9": "P10"}})

,orden_id,producto_id,importe,producto
0,101,P5,50.0,Escáner
1,102,P8,75.0,Proyector
2,101,P5,50.0,Escáner
3,103,P10,90.0,NaN
4,102,P8,75.0,Proyector


In [44]:
# …o en todo el DataFrame
ordenes.replace(50.0, 55.0)

,orden_id,producto_id,importe,producto
0,101,P5,55.0,Escáner
1,102,P8,75.0,Proyector
2,101,P5,55.0,Escáner
3,103,P9,90.0,NaN
4,102,P8,75.0,Proyector


## 4. Operaciones vectorizadas con cadenas

Los métodos de cadena de Python (`.lower()`, `.strip()`…) actúan sobre un **string**. Aplicarlos con un bucle sobre una columna es lento. Pandas ofrece el **accesor `.str`**, que aplica la operación a todos los elementos de la Serie a la vez (**vectorización**) y devuelve una nueva Serie.

```python
serie.str.<método>(...)
```

### 4.1 Limpieza básica

In [45]:
contactos = pd.DataFrame({
    "email": [" alice@email.com", "BOB@email.com ", "Charlie@email.com "],
    "nombre": ["Alice Smith", "Bob JONes", "charlie brown"],
    "codigo": ["P-101", "P-200", "P-101"],
})
contactos

,email,nombre,codigo
0,alice@email.com,Alice Smith,P-101
1,BOB@email.com,Bob JONes,P-200
2,Charlie@email.com,charlie brown,P-101


| Operación | Métodos |
|---|---|
| Mayúsculas/minúsculas | `.str.lower()`, `.str.upper()`, `.str.title()`, `.str.capitalize()` |
| Espacios | `.str.strip()`, `.str.lstrip()`, `.str.rstrip()` |
| Subcadenas | `.str[inicio:fin]`, `.str.slice()` |
| Longitud | `.str.len()` |
| Concatenación | operador `+` o `.str.cat()` |

In [46]:
limpio = contactos.copy()
limpio["email"] = limpio["email"].str.strip().str.lower()   # se pueden encadenar
limpio["nombre"] = limpio["nombre"].str.title()
limpio["inicial"] = limpio["nombre"].str[0]
limpio["etiqueta"] = limpio["nombre"] + " (" + limpio["codigo"] + ")"
limpio

,email,nombre,codigo,inicial,etiqueta
0,alice@email.com,Alice Smith,P-101,A,Alice Smith (P-101)
1,bob@email.com,Bob Jones,P-200,B,Bob Jones (P-200)
2,charlie@email.com,Charlie Brown,P-101,C,Charlie Brown (P-101)


### 4.2 Expresiones regulares

El accesor `.str` acepta **expresiones regulares** (regex) para buscar, extraer y sustituir patrones.

| Método | Devuelve | Uso típico |
|---|---|---|
| `.str.contains(patrón)` | Serie booleana | Filtrar filas |
| `.str.extract(patrón)` | Columnas con los **grupos de captura** `( )` | Extraer partes de un código |
| `.str.replace(patrón, nuevo, regex=True)` | Serie de texto | Limpiar/normalizar |
| `.str.match(patrón)` | Serie booleana | ¿Empieza por el patrón? |
| `.str.findall(patrón)` | Listas | Todas las coincidencias |

Recordatorio de regex: `\d` dígito, `[A-Z]` mayúscula, `{3}` exactamente 3 veces, `?` opcional, `^` inicio, `$` final, `( )` grupo de captura. Usa siempre cadenas *raw* `r"..."`.

In [47]:
productos = pd.DataFrame({
    "codigo": ["P-101A", "P-200B", "P-101C", "X-300D", "P-101E"],
    "descripcion": ["Core Processor 101", "Accessory Kit 200", "Core Processor 101 V2",
                    "External Drive 300", "core processor 101"],
})
productos

,codigo,descripcion
0,P-101A,Core Processor 101
1,P-200B,Accessory Kit 200
2,P-101C,Core Processor 101 V2
3,X-300D,External Drive 300
4,P-101E,core processor 101


In [48]:
# Filtrar: descripción contiene 'core processor' (sin distinguir mayúsculas)
productos[productos["descripcion"].str.contains("core processor", case=False)]

,codigo,descripcion
0,P-101A,Core Processor 101
2,P-101C,Core Processor 101 V2
4,P-101E,core processor 101


In [49]:
# Extraer: los tres dígitos y la letra final como dos columnas (grupos con nombre)
productos["codigo"].str.extract(r"-(?P<numero>\d{3})(?P<version>[A-Z])$")

,numero,version
0,101,A
1,200,B
2,101,C
3,300,D
4,101,E


In [50]:
# Reemplazar: quitar la letra final del código
productos["codigo"].str.replace(r"[A-Z]$", "", regex=True)

0    P-101
1    P-200
2    P-101
3    X-300
4    P-101
Name: codigo, dtype: str

### 4.3 Separar cadenas (tokenización): `str.split()`

`str.split(sep)` divide cada cadena y devuelve una lista. Con `expand=True` cada fragmento se convierte en una **columna**, lo que permite descomponer códigos o nombres compuestos.

In [51]:
productos["descripcion"].str.split()        # lista de palabras (tokens)

0        [Core, Processor, 101]
1         [Accessory, Kit, 200]
2    [Core, Processor, 101, V2]
3        [External, Drive, 300]
4        [core, processor, 101]
Name: descripcion, dtype: object

In [52]:
productos[["prefijo", "sufijo"]] = productos["codigo"].str.split("-", expand=True)
productos

,codigo,descripcion,prefijo,sufijo
0,P-101A,Core Processor 101,P,101A
1,P-200B,Accessory Kit 200,P,200B
2,P-101C,Core Processor 101 V2,P,101C
3,X-300D,External Drive 300,X,300D
4,P-101E,core processor 101,P,101E


## 5. Combinación de DataFrames: concat, merge y join

| Herramienta | Combina según… | Equivalente |
|---|---|---|
| `pd.concat()` | Posición / índice (apilar) | `UNION` (filas) |
| `pd.merge()` | Columnas clave | `JOIN` de SQL |
| `df.join()` | El índice | `JOIN` sobre la clave primaria |

### 5.1 Concatenación: `pd.concat()`

Apila DataFrames **verticalmente** (`axis=0`, añade filas) u **horizontalmente** (`axis=1`, añade columnas alineando por índice).

In [53]:
df1 = pd.DataFrame({"A": [1, 2], "B": [3, 4]}, index=["a", "b"])
df2 = pd.DataFrame({"A": [5, 6], "B": [7, 8]}, index=["c", "d"])
pd.concat([df1, df2])                          # axis=0 por defecto

,A,B
a,1,3
b,2,4
c,5,7
d,6,8


In [54]:
pd.concat([df1, df2], ignore_index=True)       # descarta los índices y renumera

,A,B
0,1,3
1,2,4
2,5,7
3,6,8


In [55]:
# Con keys= se crea un MultiIndex que recuerda de dónde viene cada fila
pd.concat([df1, df2], keys=["enero", "febrero"])

A  B
enero   a  1  3
        b  2  4
febrero c  5  7
        d  6  8

In [56]:
df3 = pd.DataFrame({"C": [9, 10], "D": [11, 12]}, index=["a", "b"])
pd.concat([df1, df3], axis=1)                  # lado a lado, alineando por índice

,A,B,C,D
a,1,3,9,11
b,2,4,10,12


Si los índices no coinciden, el parámetro `join` decide qué hacer:
- `join='outer'` (por defecto): se conservan todos los índices; los huecos quedan `NaN`.
- `join='inner'`: sólo los índices comunes a todos.

In [57]:
df4 = pd.DataFrame({"E": [13]}, index=["e"])
pd.concat([df1, df4], axis=1, join="outer")

,A,B,E
a,1.0,3.0,NaN
b,2.0,4.0,NaN
e,NaN,NaN,13.0


In [58]:
pd.concat([df1, df4], axis=1, join="inner")    # sin índices comunes → vacío

,A,B,E


### 5.2 Fusión estilo base de datos: `pd.merge()`

Sintaxis básica: `pd.merge(izquierdo, derecho, how=..., on=...)`

| `how=` | Filas resultado | SQL |
|---|---|---|
| `'inner'` (defecto) | Sólo claves presentes en **ambos** | `INNER JOIN` |
| `'left'` | Todas las del izquierdo | `LEFT JOIN` |
| `'right'` | Todas las del derecho | `RIGHT JOIN` |
| `'outer'` | Todas las de ambos | `FULL OUTER JOIN` |

Las columnas sin correspondencia se rellenan con `NaN`.

In [59]:
pedidos = pd.DataFrame({
    "cliente_id": [101, 102, 103, 104],
    "producto": ["A", "B", "C", "D"],
    "cantidad": [5, 2, 8, 1],
})
nombres = pd.DataFrame({
    "ID": [101, 102, 104, 105],          # falta el 103 y sobra el 105
    "nombre": ["Alice", "Bob", "David", "Eve"],
})
display(pedidos, nombres)

,cliente_id,producto,cantidad
0,101,A,5
1,102,B,2
2,103,C,8
3,104,D,1


,ID,nombre
0,101,Alice
1,102,Bob
2,104,David
3,105,Eve


**Clave con el mismo nombre → `on`**

In [60]:
pd.merge(pedidos, nombres.rename(columns={"ID": "cliente_id"}), on="cliente_id")

,cliente_id,producto,cantidad,nombre
0,101,A,5,Alice
1,102,B,2,Bob
2,104,D,1,David


**Clave con nombres distintos → `left_on` / `right_on`** (se conservan ambas columnas)

In [61]:
pd.merge(pedidos, nombres, how="left", left_on="cliente_id", right_on="ID")

,cliente_id,producto,cantidad,ID,nombre
0,101,A,5,101.0,Alice
1,102,B,2,102.0,Bob
2,103,C,8,NaN,NaN
3,104,D,1,104.0,David


### 5.3 Los cuatro tipos de join

```
   pedidos (izq.)        nombres (der.)
   101 102 103 104       101 102 104 105

   inner → 101 102 104
   left  → 101 102 103 104
   right → 101 102 104 105
   outer → 101 102 103 104 105
```

El parámetro `indicator=True` añade la columna `_merge`, que dice de qué tabla procede cada fila: muy útil para depurar.

In [62]:
for como in ["inner", "left", "right", "outer"]:
    r = pd.merge(pedidos, nombres, how=como, left_on="cliente_id", right_on="ID")
    print(f"{como:>5}: {len(r)} filas")

inner: 3 filas
 left: 4 filas
right: 4 filas
outer: 5 filas


In [63]:
pd.merge(pedidos, nombres, how="outer", left_on="cliente_id", right_on="ID", indicator=True)

,cliente_id,producto,cantidad,ID,nombre,_merge
0,101.0,A,5.0,101.0,Alice,both
1,102.0,B,2.0,102.0,Bob,both
2,103.0,C,8.0,NaN,NaN,left_only
3,104.0,D,1.0,104.0,David,both
4,NaN,NaN,NaN,105.0,Eve,right_only


> **Columnas repetidas.** Si ambos DataFrames tienen columnas con el mismo nombre que no son clave, pandas añade los sufijos `_x` y `_y`. Se pueden personalizar con `suffixes=("_izq", "_der")`.
>
> **Comprobar la cardinalidad.** `validate="one_to_one"`, `"one_to_many"` o `"many_to_one"` lanza un error si las claves no cumplen la relación esperada: evita duplicaciones silenciosas.

### 5.4 Unión por índice: `df.join()`

Es un atajo de `merge` cuando la clave es el **índice**. Por defecto hace `how='left'`.

In [64]:
ciudades = pd.DataFrame({"ciudad": ["Valladolid", "León", "Burgos"]}, index=[101, 102, 104])
facturas = pd.DataFrame({"importe": [500, 700, 800]}, index=[101, 104, 105])
ciudades.join(facturas)                 # left join por índice

,ciudad,importe
101,Valladolid,500.0
102,León,NaN
104,Burgos,700.0


In [65]:
# Equivalente con merge
pd.merge(ciudades, facturas, left_index=True, right_index=True, how="left")

,ciudad,importe
101,Valladolid,500.0
102,León,NaN
104,Burgos,700.0


| | `.join()` | `pd.merge()` |
|---|---|---|
| Clave por defecto | Índice | Columnas comunes |
| `how` por defecto | `'left'` | `'inner'` |
| Cuándo usarlo | Combinaciones por índice, sintaxis corta | Cualquier caso; más flexible |

## 6. Reestructuración: pivot, melt, stack y unstack

Un mismo conjunto de datos puede expresarse en dos formatos:

- **Largo** (*long / tidy*): una fila por observación; las variables categóricas están en columnas. Es el que prefieren las librerías de modelado y visualización (seaborn, statsmodels, scikit-learn).
- **Ancho** (*wide*): los valores de una categoría se convierten en columnas. Es el formato típico de informes y hojas de cálculo.

```
            pivot / pivot_table / unstack
   LARGO  ─────────────────────────────────▶  ANCHO
          ◀─────────────────────────────────
                  melt / stack
```

### 6.1 `pivot()`: reestructurar sin agregar

Necesita tres argumentos: `index` (filas), `columns` (cuyos valores pasan a ser columnas) y `values` (contenido de las celdas). **Cada combinación índice-columna debe ser única.**

In [66]:
largo = pd.DataFrame({
    "trimestre": ["T1", "T1", "T2", "T2", "T1", "T2"],
    "region": ["Este", "Oeste", "Este", "Oeste", "Este", "Oeste"],
    "producto": ["A", "A", "B", "B", "C", "C"],
    "ventas": [100, 150, 120, 180, 200, 250],
})
largo

,trimestre,region,producto,ventas
0,T1,Este,A,100
1,T1,Oeste,A,150
2,T2,Este,B,120
3,T2,Oeste,B,180
4,T1,Este,C,200
5,T2,Oeste,C,250


In [67]:
# Combinación (producto, region) única → pivot funciona
largo.pivot(index="producto", columns="region", values="ventas")

region,Este,Oeste
producto,,
A,100,150
B,120,180
C,200,250


In [68]:
# Combinación (trimestre, region) repetida → pivot no sabe qué valor poner
try:
    largo.pivot(index="trimestre", columns="region", values="ventas")
except ValueError as e:
    print("ValueError:", e)

ValueError: Index contains duplicate entries, cannot reshape


### 6.2 `pivot_table()`: reestructurar agregando

Equivale a las **tablas dinámicas** de una hoja de cálculo: cuando hay combinaciones repetidas, las resume con `aggfunc`.

| Parámetro | Significado |
|---|---|
| `aggfunc` | Función de agregación (`'mean'` por defecto, `'sum'`, `'count'`, lista…) |
| `fill_value` | Valor para las celdas vacías |
| `margins=True` | Añade totales de fila y columna |

> ⚠️ **Cambio respecto a versiones anteriores:** en anteriores versiones se usaba `aggfunc=np.sum`. Hoy se recomienda usar el nombre como cadena (`'sum'`); pasar funciones de NumPy genera avisos de obsolescencia.

In [69]:
pd.pivot_table(largo, index="trimestre", columns="region",
               values="ventas", aggfunc="sum")

region,Este,Oeste
trimestre,,
T1,300,150
T2,120,430


In [70]:
pd.pivot_table(largo, index="region", columns="trimestre",
               values="ventas", aggfunc=["sum", "mean"], margins=True)

sum              mean                        
trimestre   T1   T2   All     T1          T2         All
region                                                  
Este       300  120   420  150.0  120.000000  140.000000
Oeste      150  430   580  150.0  215.000000  193.333333
All        450  550  1000  150.0  183.333333  166.666667

Con varias funciones el resultado tiene **columnas jerárquicas** (MultiIndex). Aquí es donde `xs(..., axis=1)` del apartado 1.5 resulta práctico:

In [71]:
tabla = pd.pivot_table(largo, index="region", columns="trimestre",
                       values="ventas", aggfunc=["sum", "mean"])
tabla.xs("T1", level="trimestre", axis=1)

,sum,mean
region,,
Este,300,150.0
Oeste,150,150.0


### 6.3 `melt()`: de ancho a largo

Es la operación inversa de pivotar: varias columnas se "funden" en dos (nombre de la variable y valor).

| Parámetro | Significado |
|---|---|
| `id_vars` | Columnas que se mantienen como identificador |
| `value_vars` | Columnas que se funden (por defecto, todas las demás) |
| `var_name` | Nombre de la nueva columna con los antiguos nombres de columna |
| `value_name` | Nombre de la nueva columna con los valores |

In [72]:
ancho = pd.DataFrame({
    "trimestre": ["T1", "T2", "T3"],
    "ventas_A": [100, 150, 120],
    "ventas_B": [80, 110, 130],
})
ancho

,trimestre,ventas_A,ventas_B
0,T1,100,80
1,T2,150,110
2,T3,120,130


In [73]:
largo2 = ancho.melt(id_vars="trimestre",
                    value_vars=["ventas_A", "ventas_B"],
                    var_name="producto", value_name="ventas")
largo2       # 3 filas × 2 columnas fundidas = 6 filas

,trimestre,producto,ventas
0,T1,ventas_A,100
1,T2,ventas_A,150
2,T3,ventas_A,120
3,T1,ventas_B,80
4,T2,ventas_B,110
5,T3,ventas_B,130


In [74]:
# Y de vuelta a ancho
largo2.pivot(index="trimestre", columns="producto", values="ventas")

producto,ventas_A,ventas_B
trimestre,,
T1,100,80
T2,150,110
T3,120,130


### 6.4 `stack()` y `unstack()`

Trabajan sobre índices jerárquicos, moviendo niveles entre filas y columnas:

- **`unstack(level)`**: un nivel del índice de **filas** pasa a ser nivel de **columnas** (por defecto, el más interior).
- **`stack(level)`**: un nivel de **columnas** pasa al índice de **filas**.

In [75]:
midx = pd.MultiIndex.from_product([["Norte", "Sur"], ["T1", "T2"]],
                                  names=["region", "trimestre"])
ingresos = pd.Series([10, 20, 30, 40], index=midx, name="ingresos").to_frame()
ingresos

ingresos
region trimestre          
Norte  T1               10
       T2               20
Sur    T1               30
       T2               40

In [76]:
desapilado = ingresos.unstack("trimestre")
desapilado

ingresos    
trimestre       T1  T2
region                
Norte           10  20
Sur             30  40

In [77]:
desapilado.stack("trimestre")       # vuelve a la forma original

ingresos
region trimestre          
Norte  T1               10
       T2               20
Sur    T1               30
       T2               40

In [78]:
ingresos.unstack("region")          # se puede desapilar cualquier nivel

ingresos    
region       Norte Sur
trimestre             
T1              10  30
T2              20  40

> Cuando los datos ya tienen un MultiIndex (por ejemplo, tras un `groupby` con varias claves), `unstack()` es la forma más directa de pasarlos a formato ancho. Lo veremos en el siguiente apartado.

## 7. Agrupación y agregación: Split-Apply-Combine

La operación más importante del análisis de datos sigue tres pasos:

1. **Split** (separar): dividir las filas en grupos según los valores de una o varias columnas.
2. **Apply** (aplicar): calcular algo dentro de cada grupo de forma independiente.
3. **Combine** (combinar): unir los resultados en una nueva estructura.

```
            ┌ Este  → [100, 200, 120, 220] → sum → 640 ┐
 df ─split─┤                                           ├─combine→ Serie {Este: 640, Oeste: 860}
            └ Oeste → [150, 250, 180, 280] → sum → 860 ┘
```

### 7.1 `groupby()`

`df.groupby(clave)` **no calcula nada**: devuelve un objeto `DataFrameGroupBy` a la espera de la operación.

In [79]:
ventas = pd.DataFrame({
    "region":   ["Este", "Oeste", "Este", "Oeste", "Este", "Oeste", "Este", "Oeste"],
    "producto": ["A", "A", "B", "B", "A", "A", "B", "B"],
    "ventas":   [100, 150, 200, 250, 120, 180, 220, 280],
    "unidades": [10, 15, 20, 25, 12, 18, 22, 28],
})
ventas

,region,producto,ventas,unidades
0,Este,A,100,10
1,Oeste,A,150,15
2,Este,B,200,20
3,Oeste,B,250,25
4,Este,A,120,12
5,Oeste,A,180,18
6,Este,B,220,22
7,Oeste,B,280,28


In [80]:
g = ventas.groupby("region")
g                                  # objeto GroupBy: todavía no hay cálculo

In [81]:
g.groups                           # qué filas pertenecen a cada grupo

{'Este': [0, 2, 4, 6], 'Oeste': [1, 3, 5, 7]}

In [82]:
ventas.groupby("region")["ventas"].sum()

region
Este     640
Oeste    860
Name: ventas, dtype: int64

In [83]:
# Varias claves → resultado con MultiIndex
ventas.groupby(["region", "producto"])["ventas"].mean()

region  producto
Este    A           110.0
        B           210.0
Oeste   A           165.0
        B           265.0
Name: ventas, dtype: float64

In [84]:
# …que se puede pasar a formato ancho con unstack()
ventas.groupby(["region", "producto"])["ventas"].mean().unstack()

producto,A,B
region,,
Este,110.0,210.0
Oeste,165.0,265.0


Funciones de agregación habituales: `sum`, `mean`, `median`, `min`, `max`, `count` (no nulos), `size` (total de filas), `std`, `var`, `nunique`, `first`, `last`.

> `as_index=False` devuelve las claves como columnas en lugar de como índice (cómodo si después se va a hacer un `merge`).

In [85]:
ventas.groupby("region", as_index=False)["ventas"].sum()

,region,ventas
0,Este,640
1,Oeste,860


### 7.2 Varias agregaciones: `agg()`

| Forma | Ejemplo |
|---|---|
| Lista de funciones | `.agg(['count', 'min', 'max'])` |
| Diccionario por columna | `.agg({'ventas': 'sum', 'unidades': 'mean'})` |
| **Agregación con nombre** (recomendada) | `.agg(total=('ventas', 'sum'))` |

In [86]:
ventas.groupby("region")["unidades"].agg(["count", "min", "max"])

,count,min,max
region,,,
Este,4,10,22
Oeste,4,15,28


In [87]:
ventas.groupby("region").agg({"ventas": "sum", "unidades": ["mean", "max"]})

ventas unidades    
          sum     mean max
region                    
Este      640     16.0  22
Oeste     860     21.5  28

In [88]:
# Agregación con nombre: controlamos el nombre de cada columna de salida
ventas.groupby("region").agg(
    ventas_total=("ventas", "sum"),
    unidades_media=("unidades", "mean"),
    n_registros=("ventas", "size"),
)

,ventas_total,unidades_media,n_registros
region,,,
Este,640,16.0,4
Oeste,860,21.5,4


### 7.3 `transform()`: resultado del mismo tamaño que el original

Mientras `agg()` **reduce** cada grupo a un valor, `transform()` calcula el estadístico del grupo y lo **devuelve a cada fila** (mismo índice y longitud que el DataFrame). Sirve para comparar cada fila con su grupo, normalizar por grupo o imputar.

In [89]:
tr = ventas.copy()
tr["media_region"] = tr.groupby("region")["ventas"].transform("mean")    # Este 160, Oeste 215
tr["diferencia"] = tr["ventas"] - tr["media_region"]
tr["pct_region"] = tr["ventas"] / tr.groupby("region")["ventas"].transform("sum") * 100
tr

,region,producto,ventas,unidades,media_region,diferencia,pct_region
0,Este,A,100,10,160.0,-60.0,15.625000
1,Oeste,A,150,15,215.0,-65.0,17.441860
2,Este,B,200,20,160.0,40.0,31.250000
3,Oeste,B,250,25,215.0,35.0,29.069767
4,Este,A,120,12,160.0,-40.0,18.750000
5,Oeste,A,180,18,215.0,-35.0,20.930233
6,Este,B,220,22,160.0,60.0,34.375000
7,Oeste,B,280,28,215.0,65.0,32.558140


In [90]:
# Puntuación z dentro de cada grupo con una función propia
tr.groupby("region")["ventas"].transform(lambda s: (s - s.mean()) / s.std()).round(2)

0   -1.02
1   -1.08
2    0.68
3    0.58
4   -0.68
5   -0.58
6    1.02
7    1.08
Name: ventas, dtype: float64

### 7.4 Imputación por grupos

Es la técnica avanzada que se anunciaba en el apartado 2.3: rellenar cada ausente con la media **de su grupo**.

In [91]:
con_nan = ventas.copy()
con_nan.loc[[2, 5], "ventas"] = np.nan
con_nan["ventas_imp"] = con_nan["ventas"].fillna(
    con_nan.groupby("region")["ventas"].transform("mean")
)
con_nan

,region,producto,ventas,unidades,ventas_imp
0,Este,A,100.0,10,100.000000
1,Oeste,A,150.0,15,150.000000
2,Este,B,NaN,20,146.666667
3,Oeste,B,250.0,25,250.000000
4,Este,A,120.0,12,120.000000
5,Oeste,A,NaN,18,226.666667
6,Este,B,220.0,22,220.000000
7,Oeste,B,280.0,28,280.000000


### 7.5 `filter()`: conservar grupos completos

La función recibe cada grupo como un DataFrame y debe devolver `True` o `False`. Se conservan **todas las filas** de los grupos que devuelven `True`.

In [92]:
# Unidades totales: Este = 10+20+12+22 = 64 ; Oeste = 15+25+18+28 = 86
ventas.groupby("region").filter(lambda grupo: grupo["unidades"].sum() > 70)

,region,producto,ventas,unidades
1,Oeste,A,150,15
3,Oeste,B,250,25
5,Oeste,A,180,18
7,Oeste,B,280,28


### 7.6 `apply()`: lógica personalizada

Pasa cada grupo completo a una función arbitraria. Es el más flexible, pero también el más lento: úsalo sólo cuando `agg`, `transform` o `filter` no basten.

> ⚠️ **Cambio respecto a versiones anteriores:** en pandas 3, la(s) columna(s) de agrupación **ya no se incluyen** en el sub-DataFrame que recibe la función (por eso en el resultado `region` aparece sólo como índice).

In [93]:
def fila_max_ventas(grupo):
    return grupo.loc[grupo["ventas"].idxmax()]

ventas.groupby("region").apply(fila_max_ventas)

,producto,ventas,unidades
region,,,
Este,B,220,22
Oeste,B,280,28


Muchas veces existe una alternativa vectorizada más eficiente. La anterior es equivalente a:

In [94]:
ventas.loc[ventas.groupby("region")["ventas"].idxmax()]

,region,producto,ventas,unidades
6,Este,B,220,22
7,Oeste,B,280,28


### 7.7 Tablas de contingencia: `pd.crosstab()`

Cuenta las frecuencias de combinación entre dos (o más) variables categóricas. Es un caso particular de `pivot_table` con `aggfunc='count'`.

In [95]:
pd.crosstab(ventas["region"], ventas["producto"])

producto,A,B
region,,
Este,2,2
Oeste,2,2


In [96]:
# Con totales y proporciones por fila
pd.crosstab(ventas["region"], ventas["producto"], margins=True, normalize="index")

producto,A,B
region,,
Este,0.5,0.5
Oeste,0.5,0.5
All,0.5,0.5


In [97]:
# Con values + aggfunc se puede agregar otra columna
pd.crosstab(ventas["region"], ventas["producto"], values=ventas["ventas"], aggfunc="sum")

producto,A,B
region,,
Este,220,420
Oeste,330,530


### Resumen: ¿qué método de GroupBy uso?

| Método | Entrada a la función | Tamaño del resultado | Uso típico |
|---|---|---|---|
| `agg()` | Cada columna de cada grupo | 1 fila por grupo | Informes, estadísticas resumen |
| `transform()` | Cada columna de cada grupo | Igual que el original | Normalizar, comparar con el grupo, imputar |
| `filter()` | Cada grupo (DataFrame) → `bool` | Subconjunto de filas originales | Descartar grupos pequeños o irrelevantes |
| `apply()` | Cada grupo (DataFrame) | Variable | Lógica que no encaja en los anteriores |

## 8. Ejercicios propuestos

Trabaja con el siguiente conjunto de datos de calificaciones (ficticio) y resuelve cada apartado en una celda nueva.

In [98]:
notas = pd.DataFrame({
    "alumno": [" ana ruiz", "LUIS PÉREZ", "marta gil ", "luis pérez", "Pablo Sanz", "ana ruiz", "Eva Mora"],
    "grupo": ["A", "B", "A", "B", "B", "A", "A"],
    "modulo": ["PIA", "PIA", "SAA", "PIA", "MIA", "PIA", "SAA"],
    "nota": [7.5, np.nan, 8.0, 6.0, 5.5, 7.5, np.nan],
    "email": ["ana@ribera.es", "luis@ribera.es", "marta@ribera.es", "luis@ribera.es",
              "pablo@ribera.es", "ana@ribera.es", "eva@ribera.es"],
})
modulos = pd.DataFrame({
    "codigo": ["PIA", "SAA", "MIA", "BDA"],
    "horas": [140, 90, 120, 100],
})
notas

,alumno,grupo,modulo,nota,email
0,ana ruiz,A,PIA,7.5,ana@ribera.es
1,LUIS PÉREZ,B,PIA,NaN,luis@ribera.es
2,marta gil,A,SAA,8.0,marta@ribera.es
3,luis pérez,B,PIA,6.0,luis@ribera.es
4,Pablo Sanz,B,MIA,5.5,pablo@ribera.es
5,ana ruiz,A,PIA,7.5,ana@ribera.es
6,Eva Mora,A,SAA,NaN,eva@ribera.es


1. **Limpieza de texto (4):** normaliza la columna `alumno` (sin espacios sobrantes y en formato título).
2. **Duplicados (3.3):** después de normalizar, ¿cuántas filas duplicadas hay? Elimínalas.
3. **Ausentes (2 y 7.4):** cuenta los ausentes por columna e imputa las notas vacías con la media de su **módulo**.
4. **Extracción (4.2):** crea la columna `usuario` con lo que aparece antes de la `@` en el email.
5. **Merge (5):** añade a cada fila las horas del módulo. ¿Qué tipo de join usarías para que aparezca también `BDA` aunque no tenga alumnos?
6. **Agrupación (7):** obtén por módulo la nota media, la máxima y el número de alumnos, con nombres de columna en español.
7. **Pivot (6):** construye una tabla con los grupos en filas, los módulos en columnas y la nota media en las celdas (rellena con 0 lo que falte).
8. **MultiIndex (1):** establece `["grupo", "modulo"]` como índice y usa `xs()` para obtener todas las filas de `PIA`.